# Competencia de embeddings

Tres rondas: **INVIERNO**, **PUTIN**, **MESSI**. Cada equipo propone una expresión libre de sumas y restas de palabras. **Gana el equipo que gane más rondas.** No se puede utilizar el objetivo en la expresión.

Cada término debe ser una sola palabra con vector disponible en el modelo. Los resultados dependen del modelo preentrenado.

In [1]:
%pip -q install spacy
!python -m spacy download es_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 20.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import ast
import numpy as np
import spacy
import pandas as pd

nlp = spacy.load("es_core_news_md")


def vector(palabra):
    doc = nlp.make_doc(palabra.lower())
    if len(doc) != 1 or not doc[0].has_vector or doc[0].vector_norm == 0:
        raise ValueError(f"Sin vector para: {palabra}")
    return np.asarray(doc[0].vector, dtype=float)


def evaluar(objetivo, expresion):
    objetivo = objetivo.strip().lower()
    def resolver(nodo):
        if isinstance(nodo, ast.Name):
            if nodo.id.casefold() == objetivo.casefold():
                raise ValueError("No se puede usar la palabra objetivo")
            return vector(nodo.id)
        if isinstance(nodo, ast.UnaryOp) and isinstance(nodo.op, (ast.UAdd, ast.USub)):
            v = resolver(nodo.operand)
            return v if isinstance(nodo.op, ast.UAdd) else -v
        if isinstance(nodo, ast.BinOp) and isinstance(nodo.op, (ast.Add, ast.Sub)):
            a, b = resolver(nodo.left), resolver(nodo.right)
            return a + b if isinstance(nodo.op, ast.Add) else a - b
        raise ValueError("Usá palabras individuales, +, - y paréntesis")

    try:
        r = resolver(ast.parse(expresion, mode="eval").body)
    except SyntaxError as e:
        raise ValueError("Expresión inválida") from e
    t = vector(objetivo)
    if np.linalg.norm(r) == 0:
        raise ValueError("La expresión produce un vector nulo")
    return float(np.dot(r, t) / (np.linalg.norm(r) * np.linalg.norm(t)))

for objetivo in ("invierno", "putin", "messi"):
    try:
        vector(objetivo)
        print(f"Objetivo disponible: {objetivo}")
    except ValueError as error:
        print(f"Revisar objetivo: {error}")

Objetivo disponible: invierno
Objetivo disponible: putin
Objetivo disponible: messi


## Probar una propuesta

Escribí el objetivo y la expresión. El resultado es la **similitud coseno**.

In [7]:
OBJETIVO = "invierno"   # invierno / putin / messi
EXPRESION = ""          # completar con la propuesta del equipo. ejemplo "sol + dia - noche"
if EXPRESION.strip():
    print(f"Similitud: {evaluar(OBJETIVO, EXPRESION):.4f}")
else:
    print("Escribí una expresión para evaluar.")

Similitud: 0.6734
